# MCP and LangChain Integration — Hands-on Lab

**Modul 2 · LLM and Agentic AI · Job Connector Program — Purwadhika Digital Technology School**

---

## Tujuan Pembelajaran

Setelah menyelesaikan lab ini, kamu akan mampu:

1. Menjelaskan **masalah integrasi** yang diselesaikan oleh MCP.
2. Membedakan **MCP Client** vs **MCP Server**, serta **Tools / Resources / Prompts**.
3. Membangun **MCP Server** pertamamu dengan `FastMCP`.
4. **Menguji server tanpa LLM** (prinsip: *test the capability before debugging the Agent*).
5. Menghubungkan server ke **LangChain** dengan `MultiServerMCPClient` dan membuat **Agent**.
6. Membaca **jejak pesan** (Human → AI → Tool → AI) untuk memahami siapa melakukan apa.
7. Menambah tool baru tanpa mengubah arsitektur Agent, dan **men-debug dari luar ke dalam**.


# Part 0 — Setup

## 0.1 Install library (versi terbaru)

Kita **tidak mengunci versi** supaya kamu mendapat versi terbaru. Notebook ini diuji dengan: `langchain 1.4.x`, `langchain-mcp-adapters 0.3.x`, `mcp 1.30.x`.

| Library | Peran |
|---|---|
| `langchain` | Framework aplikasi AI (`create_agent`, tools, messages) |
| `langchain-openai` | Koneksi ke model OpenAI |
| `mcp` | SDK resmi MCP (membuat **server** dengan `FastMCP`) |
| `langchain-mcp-adapters` | **Jembatan** MCP ↔ LangChain (berisi `MultiServerMCPClient`) |
| `python-dotenv` | Membaca API key dari file `.env` |

In [ ]:
%pip install -U -q langchain langchain-openai langchain-mcp-adapters mcp python-dotenv

## 0.2 Cek versi yang terpasang

> 💡 Kalau setelah install ada pesan *restart runtime* (Google Colab), restart dulu lalu jalankan cell dari atas lagi.

In [ ]:
from importlib.metadata import version

for pkg in ["langchain", "langchain-core", "langchain-openai", "langchain-mcp-adapters", "mcp"]:
    print(f"{pkg:25s} {version(pkg)}")

## 0.3 Siapkan API Key

Urutan yang dicoba: **Colab Secrets** → file **`.env`** → **ketik manual** (tidak tampil di layar).

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata          # jika di Google Colab
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()                              # jika ada file .env
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Masukkan OPENAI_API_KEY: ")

print("API key tersedia:", bool(os.getenv("OPENAI_API_KEY")))

## 0.4 Inisialisasi model

`temperature=0` agar jawaban lebih konsisten (penting saat belajar & debugging).

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

print(model.invoke("Balas dengan satu kata saja: siap?").content)

## 0.5 Helper untuk "mengintip" kerja Agent

Kita butuh dua fungsi kecil agar jejak kerja Agent mudah dibaca:

| Fungsi | INPUT | PROSES | OUTPUT |
|---|---|---|---|
| `to_text(content)` | `content` pesan (bisa `str` atau list *content block*) | Gabungkan semua teks | Satu `str` |
| `show_trace(result)` | hasil `agent.invoke(...)` | Loop semua pesan, cetak sesuai perannya | Jejak User → LLM → Tool → LLM |
| `ask(agent, question)` | Agent + pertanyaan | `await agent.ainvoke(...)` lalu `show_trace` | `result` |


> **Kenapa ada `async`?** Berbicara dengan MCP Server itu mirip pelayan yang menunggu masakan matang: ia tidak berdiri diam, ia boleh melayani hal lain. Di Python, ini ditulis dengan `async`/`await`. **Di Jupyter/Colab kamu cukup menulis `await` langsung di dalam cell.** (Di file `.py` biasa, bungkus dengan `asyncio.run(main())`.)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def to_text(content):
    """Ubah content pesan (str atau list of content blocks) menjadi satu string."""
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        parts.append(block.get("text", "") if isinstance(block, dict) else str(block))
    return " ".join(parts).strip()


def show_trace(result):
    """Cetak jejak kerja Agent: siapa memutuskan, siapa mengeksekusi, apa hasilnya."""
    for msg in result["messages"]:
        if isinstance(msg, HumanMessage):
            print(f"👤 HUMAN : {to_text(msg.content)}")
        elif isinstance(msg, AIMessage):
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"🧠 AGENT : memutuskan memanggil  {tc['name']}({tc['args']})")
            else:
                print(f"🧠 AGENT (jawaban akhir): {to_text(msg.content)}")
        elif isinstance(msg, ToolMessage):
            print(f"🔧 TOOL  : {msg.name} → {to_text(msg.content)}")


async def ask(agent, question, show=True):
    """Tanya Agent (versi async) dan tampilkan jejaknya."""
    result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
    if show:
        show_trace(result)
    return result

# Part 1 — Quick Review: Tool & Agent, lalu *Integration Problem*

> **Tool** = kemampuan yang bisa dipanggil aplikasi AI untuk melakukan aksi / mengambil informasi.
> **Agent** = pengambil keputusan yang menentukan **kapan** dan **tool mana** yang dipakai.

```
User → Agent → (pilih tool) → Tool → Result → Agent → Final Answer
```

Dalam restoran: **pelayan (Agent)** menerima pesanan, memutuskan stasiun mana yang dipakai, mengambil hidangan, lalu menyajikannya ke tamu.

## 1.1 Direct tool — cara yang sudah kamu kenal

| | |
|---|---|
| **INPUT** | Fungsi Python `add_direct` + dekorator `@tool` |
| **PROSES** | `create_agent(model, tools=[...])` membungkusnya menjadi Agent |
| **OUTPUT** | Agent yang bisa memanggil `add_direct` saat dibutuhkan |

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def add_direct(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


direct_agent = create_agent(
    model=model,
    tools=[add_direct],
    system_prompt="Kamu asisten matematika. Selalu gunakan tool untuk berhitung.",
)

result = direct_agent.invoke({"messages": [{"role": "user", "content": "Berapa 25 + 4?"}]})
show_trace(result)

Tool-nya **hidup di dalam kode aplikasi**. Bagus untuk aplikasi kecil. Tapi...

## 1.2 Integration Problem: bagaimana kalau butuh 20 tool dan banyak aplikasi?

Bayangkan perusahaanmu punya **20 tool** (Database, Slack, GitHub, Google Drive, CRM, Payment, ...) dan **beberapa aplikasi AI** (chatbot CS, bot internal, dashboard analitik, ...).

Tanpa standar, **setiap pasangan (aplikasi, tool)** butuh integrasi sendiri: interface, autentikasi, format data, dan pola implementasi yang berbeda-beda.

Mari kita hitung "rasa sakitnya":

In [ ]:
total_tools = 20

print(f"{'Jumlah AI App':>14} | {'Tanpa standar (App × Tool)':>27} | {'Dengan MCP (App + Tool)':>24}")
print("-" * 72)
for apps in [1, 2, 5, 10]:
    print(f"{apps:>14} | {apps * total_tools:>27} | {apps + total_tools:>24}")

**Cara membaca output:**
- *Tanpa standar*: 5 aplikasi × 20 tool = **100 integrasi custom** yang harus dibuat dan dirawat.
- *Dengan MCP*: tiap aplikasi cukup punya **1 MCP Client**, tiap tool cukup punya **1 MCP Server** → **25** komponen.

> **Analogi USB:** dulu setiap perangkat punya colokan sendiri (printer, mouse, kamera). Setelah ada USB, satu jenis colokan dipakai semua perangkat. **MCP adalah "USB untuk aplikasi AI".**

Masalahnya bergeser dari *"Bisakah Agent-ku memakai tool?"* menjadi **"Bagaimana menstandarkan komunikasi antara aplikasi AI dan kemampuan eksternal?"**

# Part 2 — Konsep MCP

**MCP (Model Context Protocol)** adalah *open protocol* yang menstandarkan cara aplikasi AI terhubung ke tool dan konteks eksternal.

> MCP adalah **protocol** — bukan LLM, bukan Agent, bukan database, bukan API gateway, bukan UI framework, dan **bukan pengganti LangChain**.

| Komponen | Perannya |
|---|---|
| **LLM** | Reasoning / generation |
| **Agent** | Decision making |
| **LangChain** | Framework aplikasi AI |
| **MCP** | Protokol komunikasi yang terstandar |
| **MCP Server** | Penyedia capability / tool |

## Arsitektur

```
 AI Application            MCP Client                 MCP Server              External System
 (LLM + Agent)  ───►  (connect, discover,  ◄─MCP─►  (expose, define,   ───►   (DB / API / Files)
                       request, receive)            execute, return)
```

- **MCP Server** *mengekspos* capability ("Ini yang saya bisa").
- **MCP Client** *terhubung & berkomunikasi* dengan server ("Saya mau pakai capability ini").

## Tiga hal yang bisa disediakan MCP Server

| Konsep | Tujuan | Analogi Restoran | Contoh |
|---|---|---|---|
| **Tools** | *Do something* | Memasak pesanan | `create_ticket()`, `send_email()`, `search_database()` |
| **Resources** | Menyediakan informasi / konteks | Papan menu & daftar alergen | `database://customers`, `file://company-policy` |
| **Prompts** | Template interaksi yang bisa dipakai ulang | Resep standar / SOP | `summarize_report`, `review_code` |

## Kenapa MCP?

1. **Standardization** — banyak aplikasi AI bicara dengan capability lewat protokol yang sama.
2. **Reusability** — satu server dipakai ulang oleh banyak aplikasi.
3. **Separation of Concerns** — AI Application ≠ Business Capability.
4. **Scalability** — menambah capability tidak perlu membangun ulang semua aplikasi.

### Quiz

**Q1.** Apakah MCP pengganti LangChain?
<details><summary>Lihat jawaban</summary>Bukan. LangChain adalah framework untuk membangun aplikasi AI (LLM + Agent + logika aplikasi). MCP adalah protokol untuk berkomunikasi dengan capability eksternal. Keduanya <b>bekerja bersama</b>.</details>

**Q2.** `file://company-policy` termasuk Tool, Resource, atau Prompt?
<details><summary>Lihat jawaban</summary><b>Resource</b> — ia menyediakan informasi/konteks, bukan melakukan aksi.</details>

**Q3.** Saat Agent menghitung `25 × 4`, siapa yang *memutuskan* memanggil tool dan siapa yang *mengeksekusi*?
<details><summary>Lihat jawaban</summary>Yang memutuskan: <b>Agent/LLM</b>. Yang mengeksekusi: <b>MCP Server</b> (lewat tool-nya). MCP Client bertugas menghubungkan & meneruskan permintaan.</details>

# Part 3 — Membangun MCP Server Pertama

**Skenario:** Math MCP Server dengan 4 tool: `add`, `subtract`, `multiply`, `divide`.

**Kenapa Math?** Agar kita fokus pada *mekanika MCP*, bukan logika bisnis yang rumit.

```
LangChain Agent → MCP Client → Math MCP Server → add / subtract / multiply / divide
```

## 3.1 Siapkan folder kerja

In [ ]:
from pathlib import Path

WORKDIR = Path("mcp_lab")
WORKDIR.mkdir(exist_ok=True)

SERVER_PATH = (WORKDIR / "math_server.py").resolve()   # path absolut -> aman dipakai di mana saja
print("Folder kerja :", WORKDIR.resolve())
print("File server  :", SERVER_PATH)

## 3.2 Tulis file server

Perintah `%%writefile` menyimpan isi cell ini sebagai file `.py`. **Server MCP harus berupa file/proses terpisah** dari notebook — itulah inti *separation of concerns*.

Baca komentar bernomor di kode, lalu jalankan.

| | |
|---|---|
| **INPUT** | Fungsi Python biasa (`add`, `subtract`, ...) |
| **PROSES** | `@mcp.tool()` mendaftarkan fungsi sebagai MCP Tool |
| **OUTPUT** | Server yang mengekspos 4 tool |

In [ ]:
%%writefile mcp_lab/math_server.py
from mcp.server.fastmcp import FastMCP

# (1) Buat server dan beri nama
mcp = FastMCP("Math Server")


# (2) Daftarkan fungsi Python biasa sebagai MCP Tool
@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return a + b


@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a.

    Args:
        a: The number to subtract from.
        b: The number to subtract.
    """
    return a - b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers.

    Args:
        a: The first number.
        b: The second number.
    """
    return a * b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b. Fails with a clear error if b is zero.

    Args:
        a: The dividend (number to be divided).
        b: The divisor (must not be zero).
    """
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


# (3) Jalankan server. 'stdio' = server berkomunikasi lewat standard input/output
if __name__ == "__main__":
    mcp.run(transport="stdio")

## 3.3 Bedah anatomi sebuah MCP Tool

Sebuah tool yang baik punya **Name, Description, Parameters, dan Return value**:

```python
@mcp.tool()                              # ← "Daftarkan fungsi ini sebagai MCP Tool"
def add(a: float, b: float) -> float:    # ← NAME = add | PARAMETERS = a, b (float) | RETURN = float
    """Add two numbers."""               # ← DESCRIPTION (docstring)
    return a + b
```

| Bagian di kode | Menjadi apa di MCP | Dibaca siapa? |
|---|---|---|
| nama fungsi `add` | **Name** | Agent (untuk memilih tool) |
| docstring `"""..."""` | **Description** | Agent (untuk paham *kapan* dipakai) |
| `a: float, b: float` | **Parameters** (skema input) | Agent (untuk tahu *apa yang diisi*) |
| `-> float` | **Return value** | Client / Agent |

> ❓ **Pertanyaan dari slide:** *bagian mana yang membantu Agent memahami apa yang dilakukan tool ini?*
> **Jawaban:** **nama + docstring + type hints**. Itu satu-satunya "mata" Agent terhadap tool-mu. Fungsinya tetap Python biasa; MCP hanya menyediakan *interface standar* di sekelilingnya.

> 🍽️ Seperti buku menu: kalau tulisannya hanya "Menu #7" tanpa deskripsi, pelayan tidak tahu kapan merekomendasikannya.

> ⚠️ **Jebakan stdio:** pada transport `stdio`, *standard output* adalah jalur komunikasi protokol. Jangan menaruh `print()` sembarangan di server — untuk log gunakan `logging` atau `print(..., file=sys.stderr)`.

## 3.4 Menjalankan server — dan kenapa tidak ada output?

Di terminal, slide menjalankannya dengan `python math_server.py`. Server `stdio` akan **hidup lalu diam menunggu pesan** dari client. Jadi "tidak ada output" itu **normal**, bukan error.

Kita buktikan: nyalakan server sebagai proses, tunggu 3 detik, cek apakah masih hidup, lalu matikan.

| | |
|---|---|
| **INPUT** | Perintah `python mcp_lab/math_server.py` |
| **PROSES** | Jalankan sebagai subprocess, tunggu 3 detik |
| **OUTPUT** | Status: masih hidup (✅) atau langsung mati (❌ + pesan error) |

In [ ]:
import subprocess, sys, time

proc = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True,
)
time.sleep(3)

if proc.poll() is None:
    print("✅ Server hidup dan sedang menunggu pesan dari client (inilah perilaku normal).")
else:
    print("❌ Server mati. Pesan error:\n", proc.stderr.read())

proc.terminate()      # matikan lagi, karena yang akan menyalakannya nanti adalah MCP Client
proc.wait()
print("Server dimatikan.")

> 🧰 **Opsional (butuh Node.js):** MCP punya alat visual bernama *Inspector* untuk mencoba server di browser:
> `npx @modelcontextprotocol/inspector python mcp_lab/math_server.py`

# Part 4 — Testing MCP Server (tanpa LLM)

> **Engineering principle:** *test the capability before debugging the Agent.*

Kalau Agent gagal memakai tool, kamu tidak tahu penyebabnya: LLM-nya? kode client-nya? server-nya? Jadi **pastikan server sehat dulu** — tanpa LLM.

Checklist dari slide:

- ✓ Server starts
- ✓ Tools are exposed
- ✓ Tool names are correct
- ✓ Tool descriptions are understandable
- ✓ Parameters are correct

Kita pakai **SDK MCP "mentah"** (`ClientSession`) supaya kamu melihat apa yang sebenarnya dilakukan sebuah MCP Client: *connect → initialize → list tools → call tool*.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def test_mcp_server(server_path, expected_tools):
    """Jalankan checklist kesehatan MCP Server (tanpa LLM)."""
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])

    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            info = await session.initialize()                       # handshake
            print(f"✅ Server starts           → '{info.serverInfo.name}'")

            listed = (await session.list_tools()).tools             # discovery
            names = [t.name for t in listed]
            print(f"✅ Tools are exposed       → {names}")

            missing = sorted(set(expected_tools) - set(names))
            mark = "✅" if not missing else "❌"
            print(f"{mark} Tool names are correct   → yang hilang: {missing or '-'}")

            no_desc = [t.name for t in listed if not (t.description or "").strip()]
            mark = "✅" if not no_desc else "❌"
            print(f"{mark} Descriptions understandable → tanpa deskripsi: {no_desc or '-'}")

            no_params = [t.name for t in listed if not t.inputSchema.get("properties")]
            mark = "✅" if not no_params else "❌"
            print(f"{mark} Parameters are correct  → tanpa parameter: {no_params or '-'}")

            return {t.name: t for t in listed}


tools_raw = await test_mcp_server(SERVER_PATH, expected_tools=["add", "subtract", "multiply", "divide"])

## 4.1 Lihat "paspor" sebuah tool: `inputSchema`

Inilah yang **sebenarnya dikirim ke LLM** — sebuah JSON Schema yang dihasilkan otomatis dari *type hints* kamu.

In [ ]:
import json

print("NAME        :", tools_raw["add"].name)
print("DESCRIPTION :", tools_raw["add"].description.splitlines()[0])
print("INPUT SCHEMA:")
print(json.dumps(tools_raw["add"].inputSchema, indent=2))

## 4.2 Panggil tool secara manual

| | |
|---|---|
| **INPUT** | nama tool + argumen, misal `("multiply", {"a": 25, "b": 4})` |
| **PROSES** | `session.call_tool(...)` — server mengeksekusi fungsinya |
| **OUTPUT** | Hasil teks + penanda `isError` |

Kita coba 3 kasus: normal, dan **kasus error** (bagi nol).

In [ ]:
async def call_raw(server_path, tool_name, arguments):
    """Panggil satu tool lewat SDK MCP mentah, tanpa LLM."""
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            res = await session.call_tool(tool_name, arguments)
            return res.content[0].text, res.isError


for tool_name, args in [("multiply", {"a": 25, "b": 4}),
                        ("divide", {"a": 100, "b": 5}),
                        ("divide", {"a": 1, "b": 0})]:
    text, is_error = await call_raw(SERVER_PATH, tool_name, args)
    status = "❌ ERROR" if is_error else "✅ OK"
    print(f"{status} | {tool_name}({args}) → {text}")

**Perhatikan:** error "bagi nol" **tidak membuat server crash** — server membalas dengan pesan error yang jelas (`isError=True`). Pesan jelas seperti ini nanti memungkinkan Agent memahami apa yang salah dan menjelaskannya ke user.

Server kita sudah sehat ✅. Sekarang baru kita sambungkan ke LangChain.

# Part 5 — MCP Client di LangChain

```
LangChain Application  →  MCP Protocol  →  MCP Server
```

Di Part 4 kita memakai SDK mentah. Di LangChain, pekerjaan yang sama dibungkus oleh **`MultiServerMCPClient`** dari paket `langchain-mcp-adapters`, dan hasilnya langsung berupa **LangChain tools**.

> 📝 **Catatan penting (beda dengan slide):** di slide tertulis `from langchain.mcp import MultiServerMCPClient`. Pada paket terbaru, import yang benar adalah:
>
> ```python
> from langchain_mcp_adapters.client import MultiServerMCPClient
> ```

## 5.1 Konfigurasi koneksi

Setiap server diberi **nama** (kunci dictionary) dan **cara menyambung** (transport).

| Kunci | Arti |
|---|---|
| `"math"` | Nama bebas untuk server ini |
| `"transport": "stdio"` | Server berupa script yang dijalankan client lewat stdin/stdout |
| `"command"` | Program yang dipakai menyalakan server. Kita pakai `sys.executable` agar **Python-nya sama dengan notebook** (lebih aman dari `"python"`) |
| `"args"` | Argumen perintah → path file server |

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [str(SERVER_PATH)],
        }
    }
)
print("Client siap. (Belum terhubung — koneksi baru dibuka saat dibutuhkan.)")

## 5.2 Discovering MCP Tools

| | |
|---|---|
| **INPUT** | Konfigurasi koneksi di atas |
| **PROSES** | `await client.get_tools()` → client menyalakan server, *handshake*, lalu meminta daftar tool |
| **OUTPUT** | List **LangChain tools** siap pakai |

In [ ]:
tools = await client.get_tools()

for t in tools:
    print(f"🔧 {t.name}")
    print(f"     deskripsi : {t.description.splitlines()[0]}")
    print(f"     parameter : {list(t.args)}")

> 🎯 **Poin kunci:** tool-tool ini **tidak kita tulis di kode Agent**. Mereka **ditemukan (discovered)** dari MCP Server.

## 5.3 Memanggil tool secara manual lewat LangChain

Sebelum memberikan tool ke Agent, kita coba panggil sendiri. Ingat: tool MCP bersifat **async**, jadi pakai `ainvoke` (bukan `invoke`).

| | |
|---|---|
| **INPUT** | `{"a": 25, "b": 4}` |
| **PROSES** | `await tool.ainvoke(...)` → client meneruskan ke server → server mengeksekusi `multiply` |
| **OUTPUT** | Hasil dalam bentuk *content blocks* |

In [ ]:
tool_by_name = {t.name: t for t in tools}

raw = await tool_by_name["multiply"].ainvoke({"a": 25, "b": 4})

print("Output mentah :", raw)
print("Sebagai teks  :", to_text(raw))

Output mentah berupa **list of content blocks** (karena MCP bisa mengembalikan teks, gambar, dll). Helper `to_text()` kita memudahkan membacanya.

### Kesalahan paling umum pemula: memanggil dengan `invoke` biasa

In [ ]:
try:
    tool_by_name["add"].invoke({"a": 1, "b": 2})        # versi sync -> tidak didukung
except NotImplementedError as e:
    print("❌ NotImplementedError:", e)
    print("👉 Solusi: gunakan  await tool.ainvoke(...)  atau  await agent.ainvoke(...)")

# Part 6 — Membuat Agent dengan MCP Tools

```
MCP Server → MCP Client → get_tools() → LangChain Tools → Agent
```

Kodenya **hampir sama persis** dengan Part 1. Satu-satunya perbedaan: `tools` kini berasal dari MCP Server, bukan fungsi lokal.

| | |
|---|---|
| **INPUT** | `model` + `tools` hasil discovery |
| **PROSES** | `create_agent(...)` |
| **OUTPUT** | `agent` yang bisa memilih & memanggil tool MCP |

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="Kamu asisten matematika. Selalu gunakan tool untuk setiap perhitungan; jangan menghitung sendiri.",
)
print("Agent dibuat dengan tools:", [t.name for t in tools])

## 6.1 Agent memanggil MCP Tools

Pertanyaan: **`What is (25 x 4) + (100 / 5)?`**

Harapan alur kerja (step-by-step):

| Langkah | INPUT | PROSES | OUTPUT |
|---|---|---|---|
| 1 | `25, 4` | `multiply(25, 4)` | `100` |
| 2 | `100, 5` | `divide(100, 5)` | `20` |
| 3 | `100, 20` | `add(100, 20)` | `120` |

Mari kita lihat apakah Agent melakukan hal yang sama:

In [ ]:
result = await ask(agent, "What is (25 x 4) + (100 / 5)?")

**Cara membaca output:**
- 🧠 baris "memutuskan memanggil" → **keputusan Agent** (LLM).
- 🔧 baris "TOOL" → **hasil eksekusi** dari MCP Server.
- 🧠 "jawaban akhir" → Agent menyusun jawaban untuk user.

> 💡 Mungkin Agent memanggil `multiply` dan `divide` **sekaligus** (paralel) karena keduanya tidak saling bergantung. `add` baru bisa dipanggil setelah keduanya selesai. Itu bukti Agent benar-benar *bernalar* tentang urutan.

# Part 7 — Behind the Process

Di balik layar, percakapan Agent adalah **daftar pesan** dengan urutan:

```
HumanMessage → AIMessage (berisi tool call) → ToolMessage → AIMessage (jawaban akhir)
```

| Langkah | Pesan | INPUT | PROSES | OUTPUT |
|---|---|---|---|---|
| 1 | `HumanMessage` | Pertanyaan user | Masuk ke Agent | Pesan pertama dalam history |
| 2 | `AIMessage` + `tool_calls` | History pesan | **LLM memutuskan** tool & argumen | Daftar `tool_calls` |
| 3 | `ToolMessage` | `tool_calls` | **MCP Server mengeksekusi** tool | Hasil tool |
| 4 | `AIMessage` | History + hasil tool | LLM menyusun jawaban | Jawaban akhir |

Kita buktikan dengan kode, bukan hanya teori:

In [ ]:
messages = result["messages"]

print("Urutan pesan:")
for i, m in enumerate(messages, start=1):
    extra = ""
    if isinstance(m, AIMessage) and m.tool_calls:
        extra = f"  → tool_calls: {[tc['name'] for tc in m.tool_calls]}"
    elif isinstance(m, ToolMessage):
        extra = f"  → dari tool '{m.name}', hasil: {to_text(m.content)}"
    print(f"  {i}. {type(m).__name__}{extra}")

## 7.1 Jawab 4 pertanyaan dari slide — dengan bukti dari data

In [ ]:
decided_by  = [tc["name"] for m in messages if isinstance(m, AIMessage) for tc in m.tool_calls]
executed_by = [m.name for m in messages if isinstance(m, ToolMessage)]
final_answer = [m for m in messages if isinstance(m, AIMessage) and not m.tool_calls][-1]

print("1) Siapa yang memutuskan memanggil tool?  → Agent / LLM")
print("   bukti (tool yang diminta Agent)        :", decided_by)
print()
print("2) Siapa yang mengeksekusi tool?          → MCP Server / tool")
print("   bukti (ToolMessage dari tool)          :", executed_by)
print()
print("3) Dari mana tool berasal?                → MCP Client (connect & discover)")
print("   bukti (tidak ada @tool di kode Agent)  :", [t.name for t in tools])
print()
print("4) Siapa yang mengembalikan jawaban akhir?→ Agent / LLM")
print("   bukti                                  :", to_text(final_answer.content))

## 7.2 Eksperimen: bagaimana kalau tool yang dibutuhkan *tidak ada*?

Server kita belum punya tool `power` (pangkat). Mari kita minta Agent menghitung **7.3 pangkat 4.1**.

Kemungkinan perilaku (hasil bisa bervariasi):
- Agent mengaku **tidak punya tool** yang sesuai, atau
- Agent **menebak/mengaproksimasi** (berbahaya untuk angka!).

Bandingkan dengan jawaban yang benar dari Python.

In [ ]:
_ = await ask(agent, "Berapa 7.3 pangkat 4.1? Gunakan tool.")

print()
print("Jawaban benar menurut Python (7.3 ** 4.1) :", round(7.3 ** 4.1, 4))

Simpan temuan ini. Di **Part 9** kita akan menambahkan tool `power` dan mengulang pertanyaan yang sama.

# Part 8 — Direct Tools vs MCP Tools

| Aspek | Direct Integration | MCP Integration |
|---|---|---|
| Setup | Lebih sederhana | Lebih banyak komponen |
| Aplikasi kecil | Sangat baik | Mungkin berlebihan |
| Banyak tool | Bisa jadi rumit | Lebih terstandar |
| Reusability | Tergantung implementasi | Lebih kuat |
| Separation | Terintegrasi erat | Lebih terpisah (*decoupled*) |
| Standardization | Spesifik aplikasi | Berbasis protokol |

> **MCP tidak otomatis lebih baik.** Pakai MCP ketika standardisasi, reuse, dan separation memang bernilai.

## 8.1 Bukti *reusability*: dua aplikasi, satu server

**App A** = kalkulator ringkas. **App B** = guru matematika yang sabar. Keduanya memakai **MCP Server yang sama** — tanpa menulis ulang satu baris pun kode tool.

| | |
|---|---|
| **INPUT** | Satu list `tools` (hasil discovery) |
| **PROSES** | Dua `create_agent` dengan *system prompt* berbeda |
| **OUTPUT** | Dua "aplikasi AI" dengan kepribadian berbeda, kemampuan hitung sama |

In [ ]:
app_a = create_agent(
    model=model, tools=tools,
    system_prompt="Kamu kalkulator. Gunakan tool untuk berhitung, jawab seringkas mungkin (satu kalimat).",
)

app_b = create_agent(
    model=model, tools=tools,
    system_prompt=("Kamu guru matematika SMP yang sabar. Gunakan tool untuk setiap perhitungan, "
                   "lalu jelaskan langkah-langkahnya dengan bahasa sederhana."),
)

question = "Sebuah toko menjual 12 buku seharga 15 ribu rupiah per buku. Berapa total harganya?"

print("=== App A (Kalkulator) ===")
_ = await ask(app_a, question)
print("\n=== App B (Guru Matematika) ===")
_ = await ask(app_b, question)

## 8.2 Kapan pakai yang mana?

| Pilih **Direct tool** jika... | Pilih **MCP** jika... |
|---|---|
| Prototype / demo cepat | Banyak tool yang terus bertambah |
| 1–3 tool, 1 aplikasi | Satu tool dipakai **banyak aplikasi / tim** |
| Tool dan aplikasi dikelola orang yang sama | Tool dikelola **tim lain** / perlu dipisah (mis. kredensial database) |
| Tidak perlu berbagi tool | Ingin mengganti aplikasi AI tanpa menulis ulang tool |

Di **Challenge 5** kamu akan berlatih mengambil keputusan ini.

# Part 9 — Menambah Tool Baru: `power`

> *Agent bisa mendapatkan kemampuan baru tanpa mengubah arsitektur dasarnya.*

Langkahnya:

| Langkah | INPUT | PROSES | OUTPUT |
|---|---|---|---|
| 1 | Fungsi `power` | Tambahkan ke file server (`@mcp.tool()`) | Server punya 5 tool |
| 2 | Client yang sama | `await client.get_tools()` lagi (discovery ulang) | List tools berisi 5 tool |
| 3 | List tools baru | Buat ulang Agent: `create_agent(...)` | Agent yang tahu `power` |

> Di dunia nyata kamu cukup menambahkan fungsi ini di editor. Di notebook, kita menyisipkannya lewat kode supaya tetap satu alur.

In [ ]:
POWER_TOOL = '''
@mcp.tool()
def power(a: float, b: float) -> float:
    """Calculate a raised to the power of b.

    Args:
        a: The base.
        b: The exponent.
    """
    return a ** b
'''

source = SERVER_PATH.read_text()
if "def power(" not in source:
    marker = 'if __name__ == "__main__":'
    source = source.replace(marker, POWER_TOOL.strip() + "\n\n\n" + marker)
    SERVER_PATH.write_text(source)
    print("Tool 'power' ditambahkan ke math_server.py")
else:
    print("Tool 'power' sudah ada di math_server.py")

### Langkah 2 — Discovery ulang

Client kita *stateless*: tiap `get_tools()` menyalakan server dari file terbaru, jadi tidak perlu restart manual. Tools lama masih ada, dan `power` muncul sebagai tool baru.

In [ ]:
tools = await client.get_tools()
print("Tools sekarang:", [t.name for t in tools])
print("Expected      : add, subtract, multiply, divide, power")

### Langkah 3 — Buat ulang Agent dengan tools terbaru

Agent menerima daftar tool **saat dibuat**, jadi setelah ada tool baru kita buat Agent lagi. Perhatikan: kode `create_agent` **sama persis** seperti sebelumnya.

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="Kamu asisten matematika. Selalu gunakan tool untuk setiap perhitungan; jangan menghitung sendiri.",
)

_ = await ask(agent, "Berapa 7.3 pangkat 4.1? Gunakan tool.")
print()
print("Jawaban benar menurut Python (7.3 ** 4.1) :", round(7.3 ** 4.1, 4))

**Bandingkan dengan eksperimen Part 7.2.** Sekarang Agent punya tool yang tepat, sehingga jawabannya akurat — tanpa mengubah satu baris pun logika Agent.

Kita juga bisa menggabungkan beberapa tool dalam satu pertanyaan:

In [ ]:
_ = await ask(agent, "Hitung 2 pangkat 10, lalu bagi hasilnya dengan 4, lalu tambahkan 6.")

# Part 10 — Debugging MCP Integration

**Prinsip: debug dari luar ke dalam.** Jangan langsung menyalahkan LLM.

Urutan pengecekan dari slide, kita jadikan **fungsi otomatis**:

| Urutan | Pertanyaan | Kalau gagal... |
|---|---|---|
| 1 | File server ada? | Periksa path |
| 2 | Server bisa hidup? | Periksa Python / environment / dependencies |
| 3 | Client bisa terhubung? | Periksa konfigurasi koneksi |
| 4 | Tool muncul (discovered)? | Periksa registrasi tool & koneksi server |
| 5 | Nama & deskripsi tool jelas? | Agent tidak akan memilih tool yang deskripsinya buruk |
| 6 | Tool menghasilkan hasil yang benar? | Periksa parameter & implementasi tool |
| 7 | Baru: Agent berjalan benar? | Periksa prompt / model |

In [ ]:
def root_cause(exc):
    """Kupas ExceptionGroup sampai ketemu penyebab aslinya."""
    while getattr(exc, "exceptions", None):
        exc = exc.exceptions[0]
    return exc


async def diagnose_mcp(name, config, smoke_tests=None):
    """Diagnosa koneksi MCP dari luar ke dalam. Berhenti di masalah pertama."""
    print(f"🩺 Diagnosa server '{name}'\n" + "-" * 60)

    # 1) File ada?
    script = config["args"][0]
    if not Path(script).exists():
        print(f"❌ [1] File server tidak ditemukan: {script}")
        print("   👉 Periksa path / typo nama file.")
        return None
    print(f"✅ [1] File server ada: {script}")

    # 2) Server bisa hidup?
    proc = subprocess.Popen([config["command"], *config["args"]],
                            stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True)
    time.sleep(3)
    alive = proc.poll() is None
    err = "" if alive else proc.stderr.read().strip()
    proc.terminate(); proc.wait()
    if not alive:
        print("❌ [2] Server langsung mati. Baris terakhir pesan error:")
        print("   " + (err.splitlines()[-1] if err else "(tidak ada pesan)"))
        print("   👉 Periksa Python / environment / dependencies / sintaks di file server.")
        return None
    print("✅ [2] Server bisa hidup")

    # 3) Client bisa terhubung & 4) tool muncul?
    try:
        tools_ = await MultiServerMCPClient({name: config}).get_tools()
    except BaseException as e:
        print(f"❌ [3] Client gagal terhubung: {type(root_cause(e)).__name__}: {root_cause(e)}")
        return None
    print("✅ [3] Client terhubung")
    if not tools_:
        print("❌ [4] Tidak ada tool yang muncul. 👉 Periksa @mcp.tool() dan koneksi.")
        return None
    print(f"✅ [4] Tool ditemukan: {[t.name for t in tools_]}")

    # 5) Deskripsi jelas?
    no_desc = [t.name for t in tools_ if not (t.description or "").strip()]
    if no_desc:
        print(f"⚠️  [5] Tool tanpa deskripsi: {no_desc}")
        print("   👉 Tambahkan docstring, kalau tidak Agent sulit memilih tool ini.")
    else:
        print("✅ [5] Semua tool punya deskripsi")

    # 6) Smoke test: panggil tool dengan nilai yang jawabannya kita tahu
    if smoke_tests:
        by_name = {t.name: t for t in tools_}
        for tool_name, (args, expected) in smoke_tests.items():
            if tool_name not in by_name:
                print(f"❌ [6] Tool '{tool_name}' tidak ada"); continue
            out = to_text(await by_name[tool_name].ainvoke(args))
            try:
                ok = abs(float(out) - expected) < 1e-9
            except ValueError:
                ok = False
            mark = "✅" if ok else "❌"
            note = "" if ok else f"   👉 harusnya {expected}"
            shown = out if len(out) <= 110 else out[:110] + "…"
            print(f"{mark} [6] {tool_name}({args}) → {shown}{note}")
    return tools_


SMOKE_TESTS = {
    "add":      ({"a": 2,  "b": 3}, 5),
    "subtract": ({"a": 10, "b": 4}, 6),
    "multiply": ({"a": 6,  "b": 7}, 42),
    "divide":   ({"a": 20, "b": 5}, 4),
    "power":    ({"a": 2,  "b": 10}, 1024),
}

math_config = {"transport": "stdio", "command": sys.executable, "args": [str(SERVER_PATH)]}
_ = await diagnose_mcp("math", math_config, SMOKE_TESTS)

Server kita sehat. Sekarang kita **sengaja bikin masalah** untuk berlatih membaca gejalanya.

## 10.1 Gejala: "Server doesn't start" (salah path)

In [ ]:
typo_config = {"transport": "stdio", "command": sys.executable,
               "args": ["mcp_lab/math_serverr.py"]}      # <- typo: 'serverr'
_ = await diagnose_mcp("typo", typo_config)

## 10.2 Gejala: "Tool execution fails"

Tool dipanggil, tapi eksekusinya bermasalah (mis. bagi nol). Perhatikan: pesan error dikembalikan sebagai **hasil tool**, sehingga Agent bisa membacanya dan menjelaskan ke user.

In [ ]:
out = await tool_by_name["divide"].ainvoke({"a": 10, "b": 0})
print("Hasil tool langsung:", to_text(out))
print()
_ = await ask(agent, "Berapa 10 dibagi 0?")

## 10.3 Gejala: "Agent doesn't select the tool" (deskripsi buruk)

Kita buat server dengan tool **tanpa docstring** dan nama tidak informatif.

In [ ]:
%%writefile mcp_lab/vague_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Vague Server")


@mcp.tool()
def calc(x: float, y: float) -> float:
    return x + y


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
vague_config = {"transport": "stdio", "command": sys.executable,
                "args": [str(WORKDIR / "vague_server.py")]}
vague_tools = await diagnose_mcp("vague", vague_config)

hasil peringatan: tool `calc` **tidak punya deskripsi**. Bagi manusia, `calc(x, y)` mungkin masih bisa ditebak; bagi Agent, ia hanya melihat nama + skema tanpa petunjuk *kapan* dipakai.

**Eksperimen (hasil bisa bervariasi):** coba tanyakan ke Agent yang hanya punya tool ini. Lalu perbaiki: ganti nama jadi `add_numbers` dan tambahkan docstring, jalankan ulang, dan bandingkan.

In [ ]:
vague_agent = create_agent(
    model=model, tools=vague_tools,
    system_prompt="Gunakan tool bila relevan.",
)
_ = await ask(vague_agent, "Tolong jumlahkan 12 dan 30.")

# Part 11 — Bonus: Resources, Prompts, Multi-Server, dan HTTP

## 11.1 Server dengan Resource & Prompt

Sejauh ini kita hanya memakai **Tools**. Sekarang kita buat server yang menyediakan **Resource** (informasi/konteks) dan **Prompt** (template yang bisa dipakai ulang).

In [ ]:
%%writefile mcp_lab/info_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Info Server")


# RESOURCE = informasi / konteks (hanya dibaca, tidak melakukan aksi)
@mcp.resource("math://constants")
def math_constants() -> str:
    """Konstanta matematika yang sering dipakai."""
    return "pi = 3.14159265\ne = 2.71828183\nphi = 1.61803399"


# PROMPT = template interaksi yang bisa dipakai ulang
@mcp.prompt()
def explain_step_by_step(question: str) -> str:
    """Template untuk menyelesaikan soal matematika secara bertahap."""
    return (
        "Selesaikan soal berikut langkah demi langkah. "
        "Gunakan tool untuk setiap perhitungan, lalu rangkum jawaban akhirnya.\n\n"
        f"Soal: {question}"
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 11.2 Satu client, dua server (multi-server)

Inilah alasan namanya **Multi**ServerMCPClient. Tools dari semua server digabung, dan Agent tidak perlu tahu tool itu berasal dari server mana.

In [ ]:
multi_client = MultiServerMCPClient(
    {
        "math": {"transport": "stdio", "command": sys.executable, "args": [str(SERVER_PATH)]},
        "info": {"transport": "stdio", "command": sys.executable, "args": [str((WORKDIR / "info_server.py").resolve())]},
    }
)

# Resources
blobs = await multi_client.get_resources("info")
for b in blobs:
    print("📄 Resource :", b.metadata["uri"])
    print(b.as_string())

# Prompts -> hasilnya list pesan yang bisa langsung diberikan ke Agent
prompt_messages = await multi_client.get_prompt(
    "info", "explain_step_by_step", arguments={"question": "(8 + 4) * 3 - 5"}
)
print("\n📝 Prompt dari server:\n", to_text(prompt_messages[0].content))

In [ ]:
multi_tools = await multi_client.get_tools()
print("Tools gabungan:", [t.name for t in multi_tools])

prompt_agent = create_agent(model=model, tools=multi_tools)
result = await prompt_agent.ainvoke({"messages": prompt_messages})
show_trace(result)

## 11.3 Preview: transport HTTP (tidak dijalankan di notebook)

`stdio` cocok untuk server **lokal** yang dinyalakan oleh client. Untuk server yang berjalan **terpisah / di jaringan**, gunakan HTTP:

**Server** (`http_server.py`):
```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math HTTP", port=8765)

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="streamable-http")      # dijalankan di terminal: python http_server.py
```

**Client:**
```python
client = MultiServerMCPClient({
    "math_http": {"transport": "streamable_http", "url": "http://127.0.0.1:8765/mcp"}
})
tools = await client.get_tools()
```

Perhatikan: hanya blok konfigurasi yang berubah — **kode Agent tetap sama**. Itulah kekuatan standar. Referensi lengkap: dokumentasi *Model Context Protocol (MCP)* di docs.langchain.com.

# Part 12 — Ringkasan

## Cheat sheet

| Tugas | Kode |
|---|---|
| Buat server | `mcp = FastMCP("Nama")` |
| Daftarkan tool | `@mcp.tool()` + docstring + type hints |
| Daftarkan resource / prompt | `@mcp.resource("uri://...")` / `@mcp.prompt()` |
| Jalankan server | `mcp.run(transport="stdio")` |
| Test tanpa LLM | `ClientSession.list_tools()` / `.call_tool()` |
| Buat client | `MultiServerMCPClient({...})` dari `langchain_mcp_adapters.client` |
| Discovery | `tools = await client.get_tools()` |
| Buat Agent | `create_agent(model=model, tools=tools)` |
| Jalankan Agent (MCP) | `await agent.ainvoke({"messages": [...]})` |

## Alur lengkap

```
Server (@mcp.tool) ─► Client (get_tools) ─► LangChain Tools ─► Agent ─► (Human→AI→Tool→AI) ─► Final Answer
```

## ⚠️ Kesalahan umum

| Gejala | Penyebab | Solusi |
|---|---|---|
| `NotImplementedError ... sync invocation` | Memakai `invoke` pada tool MCP | Pakai `await ...ainvoke(...)` |
| `ModuleNotFoundError: langchain.mcp` | Import dari slide lama | `from langchain_mcp_adapters.client import MultiServerMCPClient` |
| Server langsung mati | Sintaks salah (mis. `@mcp.tool` tanpa `()`) | Lihat pesan error lewat `diagnose_mcp()` |
| Agent tidak memakai tool | Deskripsi/nama tool buruk | Perjelas docstring & nama |
| Tool baru tidak terpakai | Agent dibuat sebelum tool ditambahkan | `get_tools()` ulang lalu `create_agent` ulang |
| Path file tidak ketemu | Path relatif / folder berbeda | Pakai `Path(...).resolve()` |

## Prinsip engineering yang kita pakai

1. **Test the capability before debugging the Agent.**
2. **Debug dari luar ke dalam.**
3. **MCP tidak otomatis lebih baik** — pakai saat standardization, reuse, dan separation bernilai.

## Challenge 1 - Tambah Dua Tool ke Math Server

**Tugas:** Tambahkan ke `math_server.py`:
1. `sqrt(x: float) -> float` — akar kuadrat; **error yang jelas** jika `x` negatif.
2. `percentage(part: float, total: float) -> float` — berapa persen `part` dari `total`; error jika `total` nol.

**Kriteria sukses:**
- [ ] `test_mcp_server(...)` menampilkan tool baru dengan ✅
- [ ] Agent menjawab benar: *"Berapa akar dari 144, lalu 36 itu berapa persen dari 144?"* → **12** dan **25%**

<details><summary>💡 Hint 1</summary>Ikuti pola Part 9: siapkan string kode tool, lalu sisipkan sebelum <code>if __name__ == "__main__":</code>.</details>
<details><summary>💡 Hint 2</summary>Akar kuadrat: <code>x ** 0.5</code> atau <code>math.sqrt(x)</code> (jangan lupa <code>import math</code> di file server). Error: <code>raise ValueError("...")</code>.</details>
<details><summary>💡 Hint 3</summary>Setelah menambah tool: <code>get_tools()</code> ulang → <code>create_agent</code> ulang.</details>

In [ ]:
# ✍️ Tulis kode Challenge 1 di sini
# Langkah: (1) siapkan kode tool  (2) sisipkan ke SERVER_PATH  (3) test_mcp_server  (4) get_tools  (5) create_agent  (6) ask

## Challenge 2 — Dua Server, Satu Agent (Unit Converter)

**Tugas:**
1. Lengkapi `converter_server.py` dengan 3 tool: `celsius_to_fahrenheit`, `km_to_miles`, `kg_to_pounds`.
2. Sambungkan **math server + converter server** ke **satu** `MultiServerMCPClient`.
3. Tanyakan: *"Suhu 36.6°C itu berapa Fahrenheit? Lalu kalikan hasilnya dengan 2."* → butuh tool dari **dua server berbeda**.

**Kriteria sukses:**
- [ ] `diagnose_mcp` untuk converter server semua ✅
- [ ] Tool dari 2 server muncul dalam satu list
- [ ] Agent memanggil `celsius_to_fahrenheit` **lalu** `multiply`; jawaban ≈ **195.76**

<details><summary>💡 Hint 1</summary>Rumus: F = C × 9/5 + 32. 1 km = 0,621371 mil. 1 kg = 2,20462 pon.</details>
<details><summary>💡 Hint 2</summary>Dictionary konfigurasi client cukup punya dua kunci, mis. <code>"math"</code> dan <code>"converter"</code>.</details>
<details><summary>💡 Hint 3</summary>Docstring harus menyebut <b>satuan</b> input dan output (mis. "Convert Celsius to Fahrenheit") agar Agent tidak salah pilih.</details>

In [ ]:
%%writefile mcp_lab/converter_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Converter Server")

# TODO 1: tool celsius_to_fahrenheit(celsius: float) -> float      (F = C * 9/5 + 32)
# TODO 2: tool km_to_miles(km: float) -> float                     (1 km = 0.621371 mil)
# TODO 3: tool kg_to_pounds(kg: float) -> float                    (1 kg = 2.20462 pon)
# Ingat: nama jelas, docstring jelas, type hints lengkap!


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
SIAP_C2 = False     # ← ubah ke True setelah TODO selesai

if SIAP_C2:
    converter_path = str((WORKDIR / "converter_server.py").resolve())

    # TODO: jalankan diagnose_mcp untuk converter server
    # TODO: buat client_2 dengan DUA server (math + converter)
    # TODO: get_tools(), create_agent(...), lalu ask(...)
    pass
else:
    print("⏳ Challenge 2 belum dikerjakan (SIAP_C2 = False)")